# CM3070 Final Project
# Model 1 - PDF Extraction & Clause Chunking

This notebook implements **Model 1** of the Contract Analysis AI pipeline: it takes a raw employment contract PDF, extracts the text with `pdfplumber`, cleans up common PDF-extraction artefacts and splits the document into clause-level chunks ready to be classified by **Model 2** (LegalBERT / zero-shot classifier).

# Cell 1 - Install Dependencies

In [ ]:
!apt-get install -y tesseract-ocr poppler-utils -qq
!pip install pdfplumber reportlab pandas pytesseract pdf2image -q

# Cell 2 - Import Libraries

In [ ]:
import pdfplumber
import re
import pandas as pd
from collections import defaultdict
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path
from google.colab import files

# Cell 3 - Get a Contract PDF

Upload employment contract PDF using the file picker below. If upload is skipped, a synthetic sample contract is generated instead so the pipeline can still be demonstrated end-to-end.

In [ ]:
def get_contract_pdf():
    print("Upload a contract PDF or press Cancel/skip to use a sample contract instead.")
    try:
        uploaded = files.upload()
        if uploaded:
            pdf_path = list(uploaded.keys())[0]
            print(f"\nUsing uploaded file: {pdf_path}")
            return pdf_path
    except Exception as e:
        print(f"No file uploaded ({e}). Falling back to sample contract.")

    print("No upload detected - generating a synthetic sample contract instead.")
    return generate_sample_contract()


def generate_sample_contract(path="sample_employment_contract.pdf"):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.units import mm
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from reportlab.lib.enums import TA_JUSTIFY

    doc = SimpleDocTemplate(path, pagesize=A4,
                             topMargin=20 * mm, bottomMargin=20 * mm,
                             leftMargin=20 * mm, rightMargin=20 * mm)
    styles = getSampleStyleSheet()
    styles['Normal'].alignment = TA_JUSTIFY

    story = [Paragraph("EMPLOYMENT AGREEMENT", styles['Title']), Spacer(1, 12),
             Paragraph("This Employment Agreement is made between Acme Technologies "
                       "Pte Ltd (\"the Company\") and Jane Tan (\"the Employee\").",
                       styles['Normal']),
             Spacer(1, 12)]

    clauses = [
        ("1. Position and Duties",
         "The Employee shall be employed as a Senior Software Engineer and shall "
         "report to the Head of Engineering. The Employee shall perform such "
         "duties as are reasonably assigned from time to time."),
        ("2. Compensation and Benefits",
         "The Employee shall receive a monthly salary of SGD 6,500, payable on "
         "the last working day of each month, together with an annual "
         "performance bonus at the discretion of the Company. The Employee "
         "shall also be entitled to medical insurance and 14 days of annual leave."),
        ("3. Probationary Period",
         "The Employee's first three months of employment shall be a "
         "probationary period, during which either party may terminate this "
         "contract with one week's notice."),
        ("4. Termination",
         "Either party may terminate this Agreement by giving one month's "
         "written notice. The Company reserves the right to terminate "
         "employment immediately for gross misconduct without notice or "
         "compensation in lieu of notice."),
        ("5. Confidentiality",
         "The Employee agrees to keep confidential all trade secrets, business "
         "strategies, client information, and proprietary data obtained during "
         "employment and shall not disclose such information to any third "
         "party, either during or after employment."),
        ("6. Intellectual Property",
         "All inventions, developments, software, and intellectual property "
         "created by the Employee in the course of employment shall be the "
         "exclusive property of the Company."),
        ("7. Non-Compete",
         "The Employee shall not, during the term of employment and for a "
         "period of 12 months thereafter, directly or indirectly engage in any "
         "business that competes with the Company within Singapore."),
        ("8. Governing Law",
         "This Agreement shall be governed by and construed in accordance with "
         "the laws of Singapore."),
    ]

    for heading, body in clauses:
        story.append(Paragraph(heading, styles['Heading2']))
        story.append(Spacer(1, 6))
        story.append(Paragraph(body, styles['Normal']))
        story.append(Spacer(1, 10))

    doc.build(story)
    print(f"Sample contract generated: {path}")
    return path


pdf_path = get_contract_pdf()

# Cell 4 - Extract Text: Native PDF Layer, with OCR Fallback for Scanned Pages

In [ ]:
def needs_ocr(pdf_path, min_chars_per_page=20):
    """Detects pages with no meaningful extractable text layer."""
    pages_needing_ocr = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if len(text.strip()) < min_chars_per_page:
                pages_needing_ocr.append(i)
    return pages_needing_ocr


def ocr_page_to_lines(image, page_num, dpi=200):
    """
    Runs Tesseract's LSTM OCR engine on a page image and groups the
    word-level output into lines with synthetic top/bottom positions.
    Positions are rescaled from OCR pixel coordinates (at the given render
    DPI) into points (1 pt = dpi/72 px) so gaps are directly comparable to
    pdfplumber's native coordinate system regardless of render resolution.
    """
    data = pytesseract.image_to_data(image, output_type=Output.DICT)
    lines = {}
    for i in range(len(data['text'])):
        word = data['text'][i].strip()
        if not word:
            continue
        key = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
        top, height = data['top'][i], data['height'][i]
        if key not in lines:
            lines[key] = {'words': [], 'top': top, 'bottom': top + height}
        lines[key]['words'].append(word)
        lines[key]['bottom'] = max(lines[key]['bottom'], top + height)

    scale = 72.0 / dpi
    ordered = sorted(lines.values(), key=lambda l: l['top'])
    out, prev_bottom = [], None
    for l in ordered:
        top_pt, bottom_pt = l['top'] * scale, l['bottom'] * scale
        gap = (top_pt - prev_bottom) if prev_bottom is not None else None
        out.append({'page': page_num, 'text': ' '.join(l['words']), 'gap_before': gap})
        prev_bottom = bottom_pt
    return out


def extract_pages(pdf_path, ocr_dpi=200):
    """
    Extracts text lines with vertical position info from each page, using
    the native PDF text layer where available and falling back to OCR
    (Model 1's third pre-trained model) for pages with no extractable text.
    Returns a list of dicts: {'page': page_num, 'text': line_text, 'gap_before': gap}.
    """
    all_lines = []
    ocr_needed_pages = set(needs_ocr(pdf_path))

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num in ocr_needed_pages:
                continue  # handled via OCR below
            try:
                lines = page.extract_text_lines()
            except Exception:
                lines = []
            prev_bottom = None
            for l in lines:
                text = l.get('text', '').strip()
                if not text:
                    continue
                gap = (l['top'] - prev_bottom) if prev_bottom is not None else None
                all_lines.append({'page': page_num, 'text': text, 'gap_before': gap})
                prev_bottom = l['bottom']

    if ocr_needed_pages:
        print(f"Pages with no extractable text layer -- running OCR: {sorted(ocr_needed_pages)}")
        images = convert_from_path(pdf_path, dpi=ocr_dpi)
        for page_num in sorted(ocr_needed_pages):
            all_lines.extend(ocr_page_to_lines(images[page_num - 1], page_num, dpi=ocr_dpi))
        all_lines.sort(key=lambda l: l['page'])

    return all_lines


extracted_lines = extract_pages(pdf_path)
print(f"Extracted {len(extracted_lines)} line(s) across "
      f"{max((l['page'] for l in extracted_lines), default=0)} page(s)")

# Cell 5 - Strip Boilerplate: Running Headers/Footers & Document Stamps

In [ ]:
_STAMP_PATTERNS = [
    re.compile(r'Updated\s+on\s+\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}', re.I),
    re.compile(r'^[A-Z]-\d+$'),  # e.g. "A-3" page/annex refs
    re.compile(r'^\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\s+\d{1,2}:\d{2}(:\d{2})?$'),  # bare timestamps
]


def strip_boilerplate(all_lines, min_page_repeats=2):
    def norm(t):
        return re.sub(r'\s+', ' ', t.strip().lower())

    page_sets = defaultdict(set)
    for rec in all_lines:
        page_sets[norm(rec['text'])].add(rec['page'])
    repeated = {t for t, pages in page_sets.items() if len(pages) >= min_page_repeats}

    out = []
    for rec in all_lines:
        text = rec['text']
        if norm(text) in repeated:
            continue
        if any(p.search(text) for p in _STAMP_PATTERNS):
            continue
        out.append(rec)
    return out


extracted_lines = strip_boilerplate(extracted_lines)
print(f"{len(extracted_lines)} line(s) remain after removing headers/footers/stamps")

# Cell 6 - Clean Lines, Detect Headings, and Chunk into Clauses

In [ ]:
def clean_line(text):
    text = re.sub(r'^\s*Page\s+\d+(\s+of\s+\d+)?\s*$', '', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text)
    return text.strip()


HEADING_PATTERNS = [
    re.compile(r'^\s*(\d{1,2})\.?\s+([A-Z][A-Za-z0-9,/&\'\-\s]{2,80})\s*$'),  # bare top-level numbers only
    re.compile(r'^\s*(ARTICLE|Article)\s+([IVXLCDM]+|\d+)[\s:.\-]*(.*)$'),
    re.compile(r'^\s*(SECTION|Section)\s+(\d+)[\s:.\-]*(.*)$'),
    re.compile(r'^\s*[A-Z][A-Z\s]{3,60}\s*$'),
]
_NON_HEADING_STARTERS = ('PLEASE ', 'NOTE ', 'NOTE:', 'WARNING', 'IMPORTANT',
                          'CAUTION', 'ATTENTION', 'DISCLAIMER')
_CLOSING_MARKERS = re.compile(r'\b(IN WITNESS WHEREOF|SIGNED AT|SIGNATURE[S]?\s*:?\s*$)\b', re.I)


def is_heading(line):
    line = line.strip()
    if not line or len(line) > 140:
        return False
    if _CLOSING_MARKERS.search(line):
        return True
    if len(line) > 100:
        return False
    if line.upper().startswith(_NON_HEADING_STARTERS):
        return False
    return any(pat.match(line) for pat in HEADING_PATTERNS)


def is_new_paragraph(line_record, prev_line_record, gap_threshold=10):
    if prev_line_record is None:
        return True
    if line_record['page'] != prev_line_record['page']:
        prev_text = prev_line_record['text'].strip()
        return prev_text[-1:] in '.!?:;' if prev_text else True
    gap = line_record['gap_before']
    return gap is not None and gap > gap_threshold


def is_placeholder_text(text, alpha_ratio_threshold=0.2):
    """True if a chunk is mostly blank fill-in dots/underscores, not real content."""
    letters = sum(1 for c in text if c.isalpha())
    return len(text) > 0 and (letters / len(text)) < alpha_ratio_threshold


def is_likely_fragment(text, min_absolute_len=10):
    """
    True if a chunk looks like an orphaned continuation of a previous
    sentence rather than a genuine standalone clause. Starting with a
    lowercase letter is the primary signal (a real clause -- however
    short -- starts with a capital letter); a small absolute-length
    floor also catches stray single-token artifacts regardless of case.
    """
    text = text.strip()
    if not text:
        return True
    if text[0].islower():
        return True
    if len(text) < min_absolute_len:
        return True
    return False


def chunk_into_clauses(all_lines, min_chunk_chars=30):
    cleaned = []
    for rec in all_lines:
        t = clean_line(rec['text'])
        if t:
            cleaned.append({**rec, 'text': t})

    heading_idx = [i for i, r in enumerate(cleaned) if is_heading(r['text'])]
    chunks = []

    def join_lines(records):
        parts = []
        for r in records:
            t = r['text']
            if parts and parts[-1].endswith('-'):
                parts[-1] = parts[-1][:-1] + t
            else:
                parts.append(t)
        return ' '.join(parts)

    if heading_idx:
        for idx, start in enumerate(heading_idx):
            end = heading_idx[idx + 1] if idx + 1 < len(heading_idx) else len(cleaned)
            heading_text = cleaned[start]['text']
            body_text = join_lines(cleaned[start + 1:end])
            page_num = cleaned[start]['page']
            if len(body_text) >= min_chunk_chars:
                chunks.append({'chunk_id': len(chunks) + 1, 'section_heading': heading_text,
                                'clause_text': body_text, 'page_number': page_num,
                                'char_count': len(body_text)})
    else:
        current, prev, groups = [], None, []
        for r in cleaned:
            if is_new_paragraph(r, prev) and current:
                groups.append(current)
                current = []
            current.append(r)
            prev = r
        if current:
            groups.append(current)
        for i, group in enumerate(groups, start=1):
            text = join_lines(group)
            if len(text) >= min_chunk_chars:
                chunks.append({'chunk_id': i, 'section_heading': None, 'clause_text': text,
                                'page_number': group[0]['page'], 'char_count': len(text)})

    chunks = [c for c in chunks if not is_placeholder_text(c['clause_text'])]

    merged = []
    for c in chunks:
        if merged and is_likely_fragment(c['clause_text']):
            merged[-1]['clause_text'] = merged[-1]['clause_text'] + ' ' + c['clause_text']
            merged[-1]['char_count'] = len(merged[-1]['clause_text'])
        else:
            merged.append(dict(c))

    for i, c in enumerate(merged, start=1):
        c['chunk_id'] = i

    return pd.DataFrame(merged)

# Cell 7 - Run Extraction Pipeline

In [ ]:
clauses_df = chunk_into_clauses(extracted_lines)

print(f"Total clauses extracted: {len(clauses_df)}\n")
pd.set_option('display.max_colwidth', 60)
print(clauses_df[['chunk_id', 'section_heading', 'page_number', 'char_count']].to_string(index=False))

# Cell 8 - Validate Extraction Quality

In [ ]:
print("=" * 60)
print("EXTRACTION QUALITY CHECK")
print("=" * 60)
print(f"Total chunks:        {len(clauses_df)}")
print(f"Average chunk length: {clauses_df['char_count'].mean():.0f} characters")
print(f"Shortest chunk:       {clauses_df['char_count'].min()} characters")
print(f"Longest chunk:        {clauses_df['char_count'].max()} characters")

shortest = clauses_df.loc[clauses_df['char_count'].idxmin()]
longest = clauses_df.loc[clauses_df['char_count'].idxmax()]

print(f"\nShortest chunk — \"{shortest['section_heading']}\":")
print(f"  {shortest['clause_text'][:150]}")
print(f"\nLongest chunk — \"{longest['section_heading']}\":")
print(f"  {longest['clause_text'][:150]}...")
print("=" * 60)

# Cell 9 - Prepare Output for Model 2 (Clause Classification)

In [ ]:
def prepare_for_model2(df, min_chars=20):
    out = df[df['char_count'] >= min_chars].copy()
    assert 'clause_text' in out.columns, "Missing clause_text column for Model 2"
    return out.reset_index(drop=True)


model2_input_df = prepare_for_model2(clauses_df)
print(f"{len(model2_input_df)} / {len(clauses_df)} chunks ready to pass to Model 2\n")
print("Preview of Model 2 input:")
print(model2_input_df[['chunk_id', 'section_heading', 'clause_text']].head(3).to_string(index=False))

# Cell 10 - Save Results

In [ ]:
output_path = "model1_extracted_clauses.csv"
model2_input_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

try:
    files.download(output_path)
except Exception as e:
    print(f"(Download skipped — not running in Colab: {e})")

# Cell 11 - Validation Checklist (Adversarial Testing)

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.enums import TA_JUSTIFY
from pdf2image import convert_from_path
from reportlab.pdfgen import canvas as pdfcanvas
from reportlab.lib.utils import ImageReader

_styles = getSampleStyleSheet()
_styles['Normal'].alignment = TA_JUSTIFY

def _build(path, story):
    doc = SimpleDocTemplate(path, pagesize=A4, topMargin=20*mm, bottomMargin=20*mm,
                             leftMargin=20*mm, rightMargin=20*mm)
    doc.build(story)

_story = [Paragraph("This Employment Agreement is entered into between Beta Logistics Pte Ltd and Mr. Ravi Kumar.", _styles['Normal']), Spacer(1,10)]
for p in [
    "The employee shall commence work on 1 March 2026 as a Warehouse Operations Manager and will be based at the company's Tuas facility, reporting directly to the Operations Director.",
    "In consideration of services rendered the employee shall be paid a monthly salary of SGD 4,200 together with transport allowance, payable by the 25th of each month via bank transfer.",
    "The first six months of service shall constitute a probationary period during which the company may terminate employment upon giving one week's notice in writing.",
    "Upon confirmation, either party may terminate this agreement by providing not less than two months notice in writing, or payment of salary in lieu thereof.",
    "The employee undertakes not to disclose any confidential business information of the company to any third party during or after the term of this agreement.",
]:
    _story.append(Paragraph(p, _styles['Normal']))
    _story.append(Spacer(1, 14))
_build("_t1_no_headings.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("5. Confidentiality", _styles['Heading2']),
          Paragraph("The Employee shall not disclose confidential information, save in the following circumstances:", _styles['Normal']),
          Paragraph("(a) where disclosure is required by law or a court order;", _styles['Normal']),
          Paragraph("(b) where the information has already entered the public domain through no fault of the Employee;", _styles['Normal']),
          Paragraph("(c) where disclosure is made to the Employee's professional advisers on a confidential basis.", _styles['Normal']),
          Spacer(1,10), Paragraph("6. Termination", _styles['Heading2']),
          Paragraph("Either party may terminate this Agreement by giving one month's written notice.", _styles['Normal'])]
_build("_t2_lettered_sublist.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("1. Position", _styles['Heading2']),
          Paragraph("The Employee is engaged as a Marketing Executive.", _styles['Normal']),
          Spacer(1,10), Paragraph("2. Compensation and Benefits", _styles['Heading2'])]
_long = ("The Employee shall receive a monthly salary of SGD 5,000 payable on the last working day of each month, in accordance with the Company's standard payroll cycle. " * 30 +
         "This final sentence deliberately sits on the following page to confirm the clause is reassembled as one chunk.")
_story.append(Paragraph(_long, _styles['Normal']))
_story.append(Spacer(1,10))
_story.append(Paragraph("3. Termination", _styles['Heading2']))
_story.append(Paragraph("Either party may terminate with one month's notice.", _styles['Normal']))
_build("_t3_page_break.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("4. Termination", _styles['Heading2']),
          Paragraph("Either party may terminate this Agreement by giving one month's written notice.", _styles['Normal']),
          Paragraph("PLEASE READ THIS CLAUSE CAREFULLY", _styles['Normal']),
          Paragraph("The Company reserves the right to terminate immediately for gross misconduct without notice.", _styles['Normal']),
          Spacer(1,10), Paragraph("5. Confidentiality", _styles['Heading2']),
          Paragraph("NOTE TO EMPLOYEE", _styles['Normal']),
          Paragraph("The Employee agrees to keep all business information confidential during and after employment.", _styles['Normal'])]
_build("_t4_allcaps_falsepositive.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("2. Compensation and Benefits", _styles['Heading2']),
          Paragraph("The Employee\u2019s monthly salary shall be S$6,500 \u2014 payable via GIRO transfer. "
                    "The Company\u2019s bonus scheme (\u201cthe Scheme\u201d) applies at management\u2019s discretion.",
                    _styles['Normal'])]
_build("_t5_encoding.pdf", _story)

_images = convert_from_path("_t1_no_headings.pdf", dpi=150)
_c = pdfcanvas.Canvas("_t6_scanned.pdf", pagesize=A4)
_w, _h = A4
for _img in _images:
    _c.drawImage(ImageReader(_img), 0, 0, width=_w, height=_h)
    _c.showPage()
_c.save()

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("8. Overtime", _styles['Heading2']),
          Paragraph("You will be entitled to overtime pay at 1.5 times the hourly basic rate for each hour of work beyond normal hours.", _styles['Normal']),
          Spacer(1,8),
          Paragraph("allowed under the Employment Act.", _styles['Normal']),
          Spacer(1,10),
          Paragraph("Sample Employment Contract Updated on 02/12/2011 12:37:31", _styles['Normal']),
          PageBreak(),
          Paragraph("9. Termination", _styles['Heading2']),
          Paragraph("Either party may terminate this Agreement by giving one month's written notice.", _styles['Normal']),
          Spacer(1,10),
          Paragraph("Sample Employment Contract Updated on 02/12/2011 12:37:31", _styles['Normal'])]
_build("_t7_footer_and_fragment.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("9. Notice Periods", _styles['Heading2']),
          Paragraph("(a) With notice given [E.g. 1 day / 1 week / 2 weeks / 1 month] (b) Without notice by paying salary in lieu of notice for the relevant period.", _styles['Normal']),
          Spacer(1,10),
          Paragraph("IN WITNESS WHEREOF the parties have signed this Agreement.", _styles['Normal']),
          Paragraph("Signed at Singapore on this day.", _styles['Normal']),
          Paragraph("(Signature) ................. Name of Employer: ________", _styles['Normal'])]
_build("_t8_signature_block.pdf", _story)

_story = [Paragraph("EMPLOYMENT AGREEMENT", _styles['Title']), Spacer(1,12),
          Paragraph("13. Amendments", _styles['Heading2']),
          Paragraph("...................................................... ......................................................", _styles['Normal']),
          Spacer(1,10),
          Paragraph("14. Governing Law", _styles['Heading2']),
          Paragraph("This Agreement shall be governed by the laws of Singapore.", _styles['Normal'])]
_build("_t9_placeholder_blank.pdf", _story)

_tests = [
    ("_t1_no_headings.pdf", "No headings at all (fallback to paragraph split)", 6),
    ("_t2_lettered_sublist.pdf", "Lettered sub-list (a)(b)(c) inside a clause", 2),
    ("_t3_page_break.pdf", "Clause spanning a genuine page break", 3),
    ("_t4_allcaps_falsepositive.pdf", "ALL-CAPS false-positive heading lines", 2),
    ("_t5_encoding.pdf", "Special characters / smart quotes / em-dash", 1),
    ("_t6_scanned.pdf", "Scanned PDF with no text layer -- recovered via OCR", 6),
    ("_t7_footer_and_fragment.pdf", "Repeated footer stamp + orphan fragment", 2),
    ("_t8_signature_block.pdf", "Unmarked signature/closing block", 2),
    ("_t9_placeholder_blank.pdf", "Blank fill-in placeholder line dropped", 1),
]

print("=" * 65)
print("MODEL 1 VALIDATION CHECKLIST (v3)")
print("=" * 65)
for fname, desc, expected_chunks in _tests:
    lines_ = extract_pages(fname)
    lines_ = strip_boilerplate(lines_)
    df_ = chunk_into_clauses(lines_)
    status = "PASS" if len(df_) == expected_chunks else "CHECK"
    print(f"[{status}] {desc}")
    print(f"        expected {expected_chunks} chunk(s), got {len(df_)}")
print("=" * 65)

# Cell 12 - Summary

In [ ]:
print("=" * 60)
print("MODEL 1 SUMMARY - PDF EXTRACTION & CHUNKING (v3)")
print("=" * 60)
print(f"Source PDF:          {pdf_path}")
print(f"Clauses extracted:   {len(clauses_df)}")
print(f"Passed to Model 2:   {len(model2_input_df)}")

print("\n--- Recent changes (found via real-contract testing) ---")
print("  1. Running headers/footers stripped via cross-page repetition detection")
print("  2. Document stamps (timestamps, annex codes) stripped via explicit patterns")
print("  3. Closing/signature blocks now split off even without a numbered heading")
print("  4. Blank fill-in placeholder chunks (dots/underscores only) are dropped")
print("  5. Short trailing fragments merge into the previous chunk instead of")
print("     standing alone as meaningless pseudo-clauses")
print("  6. OCR fallback (Tesseract LSTM engine) recovers scanned/image-only PDFs")
print("     that previously produced zero output -- this is Model 1's second")
print("     pre-trained model, alongside pdfplumber's rule-based extraction")

print("\n--- Known limitations ---")
print("  - Heading detection is regex/pattern based, not learned")
print("  - OCR quality depends on scan clarity/resolution; low-quality scans may")
print("    still produce noisy or incomplete text")
print("  - Short-fragment merging (starts-lowercase / <10 chars) could in principle")
print("    still merge a genuinely short real clause into its neighbour")
print("  - ALL-CAPS false-positive stoplist is not exhaustive")

print("\n--- Next steps ---")
print("  - Feed model2_input_df into Model 2's classify_clauses() function")
print("  - Test against a full narrative contract (not just a KETs-style form)")
print("    to exercise confidentiality/non-compete/IP clause extraction")
print("=" * 60)